In [ ]:
# Cell 1 — Install
!pip install -q "openenv-core[core]>=0.2.1"
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl transformers datasets accelerate peft
!pip install -q torch-geometric
!git clone https://huggingface.co/spaces/NeuralHU/forge-rl /content/forge-rl

import sys, os, torch
sys.path.insert(0, '/content/forge-rl')
os.chdir('/content/forge-rl')

if not torch.cuda.is_available():
    raise RuntimeError('NO GPU DETECTED. Runtime -> Change runtime type -> T4 GPU')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print('Setup complete.')

In [ ]:
# Cell 2 — Load model
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-3B-Instruct',
    max_seq_length=1024,
    load_in_4bit=True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0, bias='none',
    use_gradient_checkpointing='unsloth',
)
print('Red model loaded.')

In [ ]:
# Cell 3 — Red reward function (maximize Blue TED)
import numpy as np
from env.forge_env import ForgeEnv, ForgeEnvConfig
from rewards.tactic_edit_dist import tactic_edit_distance
from env.primitives import PrimitiveType

RED_PROMPT = ('You are a RED TEAM adversarial agent. Your goal is to craft a '
              'misinformation chain that defeats Blue Team forensic detection.\n'
              'Use these primitives: SOURCE_LAUNDER, TEMPORAL_SHIFT, '
              'ENTITY_SUBSTITUTE, QUOTE_FABRICATE, CONTEXT_STRIP, '
              'CITATION_FORGE, NETWORK_AMPLIFY, SATIRE_REFRAME\n\n'
              'Claim: {claim}\n\n'
              'Output a chain of 2-4 primitives that will evade detection:')

PRIMITIVE_NAMES = [p.value for p in PrimitiveType]

def _extract_chain(text):
    found = []
    for name in PRIMITIVE_NAMES:
        if name in text.upper():
            found.append(name)
    return found[:4]

def _safe_step(env, action):
    result = env.step(action)
    if isinstance(result, tuple):
        if len(result) == 5:
            _, reward, terminated, truncated, _ = result
            return float(reward), bool(terminated or truncated)
        elif len(result) == 4:
            _, reward, terminated, _ = result
            return float(reward), bool(terminated)
        elif len(result) == 2:
            _, reward = result
            return float(reward), False
    return float(getattr(result, 'reward', 0.0)), bool(getattr(result, 'done', False))

def _safe_reset(env):
    result = env.reset()
    if isinstance(result, tuple) and len(result) == 2:
        return result
    return result, {}

def red_reward_fn(prompts, completions, true_chains=None, **kwargs):
    rewards = []
    for i, completion in enumerate(completions):
        text = (completion[-1]['content'] if isinstance(completion, list) else str(completion))
        try:
            predicted = _extract_chain(text)
            true = true_chains[i] if true_chains and i < len(true_chains) else []
            # Red reward = TED (high TED = Blue agent confused)
            ted = tactic_edit_distance(predicted, true)
            rewards.append(float(np.clip(ted, 0.001, 0.999)))
        except Exception as e:
            print(f'[red_reward_fn] episode {i} error: {e}')
            rewards.append(0.001)
    return rewards

print('red_reward_fn ready.')

In [ ]:
# Cell 4 — Dataset
from datasets import Dataset
import random

TASK_NAMES = ['fabricated_stats', 'out_of_context', 'coordinated_campaign',
              'satire_news', 'verified_fact', 'politifact_liar',
              'image_forensics', 'sec_fraud']

def _get_claim_and_chain(task, seed):
    try:
        env = ForgeEnv(ForgeEnvConfig(budget=3, seed=seed))
        _, info = _safe_reset(env)
        claim = None
        for attr in ('_claim_text', 'claim_text'):
            val = getattr(env, attr, None)
            if val: claim = str(val); break
        if claim is None and info:
            claim = info.get('claim_text', f'Claim #{seed}')
        true_chain = [str(p.value) if hasattr(p, 'value') else str(p)
                      for p in info.get('true_chain', [])] if info else []
        return claim or f'Claim #{seed}', true_chain
    except:
        return f'Unverified claim #{seed} in domain: {task}', []

random.seed(42)
prompts, true_chains = [], []
for i in range(200):
    claim, chain = _get_claim_and_chain(TASK_NAMES[i % len(TASK_NAMES)], seed=i)
    prompts.append([{'role': 'user', 'content': RED_PROMPT.format(claim=claim)}])
    true_chains.append(chain)

dataset = Dataset.from_dict({'prompt': prompts, 'true_chains': true_chains})
print(f'Red dataset: {len(dataset)} rows')

In [ ]:
# Cell 5 — GRPO Training (Red)
from trl import GRPOConfig, GRPOTrainer

config = GRPOConfig(
    output_dir='./red-grpo',
    max_steps=150,
    num_generations=4,
    max_completion_length=128,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    save_steps=50,
    logging_steps=5,
    report_to='none',
    warmup_ratio=0.1,
)

try:
    trainer = GRPOTrainer(model=model, reward_funcs=red_reward_fn, args=config,
                          train_dataset=dataset, processing_class=tokenizer)
except TypeError:
    trainer = GRPOTrainer(model=model, reward_funcs=red_reward_fn, args=config,
                          train_dataset=dataset, tokenizer=tokenizer)

print('Starting Red GRPO training (max 150 steps)...')
trainer.train()
print('Training complete.')

In [ ]:
# Cell 6 — Plot
import matplotlib.pyplot as plt
import matplotlib; matplotlib.use('Agg')
import os
os.makedirs('results', exist_ok=True)

steps, rew = [], []
for l in trainer.state.log_history:
    for key in ('rewards/mean', 'reward/mean', 'reward'):
        if key in l:
            steps.append(l['step']); rew.append(l[key]); break

fig, ax = plt.subplots(figsize=(10, 5))
if steps: ax.plot(steps, rew, 'r-', linewidth=2.5, label='Red TED reward')
ax.set_xlabel('Training Step'); ax.set_ylabel('Mean TED Reward')
ax.set_title('FORGE-MA: Red Team Adversarial Training via GRPO')
ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.savefig('results/red_reward_curve.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: results/red_reward_curve.png')